In [ ]:
import sys
import os

# 1. Rileva se l'esecuzione avviene su Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("⚡ Esecuzione su Google Colab.")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # URL del tuo repository GitHub e percorso in Colab
    REPO_URL = "https://github.com/giacomosalvatori03/3D_Perception.git"
    REPO_DIR = "/content/3D_Perception"
    
    # Se la repo non è ancora presente su Colab la clona, altrimenti scarica gli aggiornamenti
    if not os.path.exists(REPO_DIR):
        print("📥 Clonazione del repository da GitHub...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("🔄 Aggiornamento del codice da GitHub...")
        %cd {REPO_DIR}
        !git pull
        
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # PERCORSO DEL DATASET SU GOOGLE DRIVE
    # Modifica questo percorso con la posizione esatta della cartella 'kitti_validation' sul tuo Drive
    DATASET_PATH = "/content/drive/MyDrive/3D_Perception/data/kitti_validation"

else:
    print("💻 Esecuzione in Locale.")
    PROJECT_ROOT = os.path.abspath('..')
    if PROJECT_ROOT not in sys.path:
        sys.path.append(PROJECT_ROOT)
    DATASET_PATH = os.path.join(PROJECT_ROOT, 'data/kitti_validation')

print(f"✅ Directory corrente: {os.getcwd()}")
print(f"✅ Dataset Path: {DATASET_PATH}")

⚡ Esecuzione su Google Colab.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Aggiornamento del codice da GitHub...
/content/3D_Perception
Already up to date.
✅ Directory corrente: /content/3D_Perception
✅ Dataset Path: /content/drive/MyDrive/3D_Perception/data/kitti_subset


In [2]:
from src import KittiDataset
from src.detectors.pillarization import Pillarizer

# Carica il primo sample dal dataset
dataset = KittiDataset(data_root=DATASET_PATH)
sample = dataset[0]
points = sample['points']

# Inizializza il Pillarizer
pillarizer = Pillarizer()
features, coords = pillarizer.process(points)

print(f"Punti LiDAR iniziali: {points.shape}")
print(f"Pillars estratti (P, MaxPoints, Features): {features.shape}")
print(f"Coordinate dei Pillars (P, [batch, y, x]): {coords.shape}")

Punti LiDAR iniziali: (115384, 4)
Pillars estratti (P, MaxPoints, Features): torch.Size([8235, 32, 9])
Coordinate dei Pillars (P, [batch, y, x]): torch.Size([8235, 3])


In [3]:
from src import KittiDataset, LidarDetector

# Inizializza dataset e detector
dataset = KittiDataset(data_root=DATASET_PATH)
detector = LidarDetector(apply_roi_filter=True, max_distance=50.0)

# Esegui la detection su un sample
sample = dataset[0]
detections = detector.detect(sample)
print(f"Predizioni generate: {len(detections)}")

⚙️ LidarDetector inizializzato su device: cuda
✅ Pesi PointPillars gia presenti in: weights/pointpillar_kitti.pth
📦 Caricamento checkpoint PyTorch da: weights/pointpillar_kitti.pth
✅ Checkpoint PointPillars caricato con successo in memoria!
Predizioni generate: 0


In [4]:
import os
import torch
from scripts.pointpillar_weights import download_pointpillars_weights

# 1. Scarica i pesi (se non già presenti)
weights_path = download_pointpillars_weights()

# 2. Verifica l'esistenza e la dimensione del file
if os.path.exists(weights_path):
    file_size_mb = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"📁 File trovato: {weights_path} ({file_size_mb:.2f} MB)")

    # 3. Carica il checkpoint con PyTorch
    print("⏳ Caricamento del checkpoint con torch.load()...")
    checkpoint = torch.load(weights_path, map_location='cpu')
    
    # 4. Ispeziona la struttura del file .pth
    print("\n✅ Checkpoint caricato con successo!")
    print(f"🔑 Chiavi principali del file: {list(checkpoint.keys())}")
    
    # Estraggo lo state_dict (i pesi reali dei layer della rete neurale)
    state_dict = checkpoint.get('state_dict', checkpoint)
    print(f"🧠 Numero totale di layer/parametri nella rete: {len(state_dict)}")
    
    # Mostra i primi 5 layer per conferma
    print("\nEsempio dei primi layer trovati:")
    for i, (key, value) in enumerate(state_dict.items()):
        if i >= 5:
            break
        print(f"  • {key}: shape {value.shape}")
else:
    print("❌ File dei pesi non trovato.")

✅ Pesi PointPillars gia presenti in: weights/pointpillar_kitti.pth
📁 File trovato: weights/pointpillar_kitti.pth (18.52 MB)
⏳ Caricamento del checkpoint con torch.load()...

✅ Checkpoint caricato con successo!
🔑 Chiavi principali del file: ['meta', 'state_dict']
🧠 Numero totale di layer/parametri nella rete: 126

Esempio dei primi layer trovati:
  • backbone.blocks.0.0.weight: shape torch.Size([64, 64, 3, 3])
  • backbone.blocks.0.1.weight: shape torch.Size([64])
  • backbone.blocks.0.1.bias: shape torch.Size([64])
  • backbone.blocks.0.1.running_mean: shape torch.Size([64])
  • backbone.blocks.0.1.running_var: shape torch.Size([64])


In [5]:
from src import KittiDataset, LidarDetector

# Inizializzazione detector
detector = LidarDetector(apply_roi_filter=True)

# Esecuzione su un sample di test
dataset = KittiDataset(data_root=DATASET_PATH)
sample = dataset[0]
detections = detector.detect(sample)

⚙️ LidarDetector inizializzato su device: cuda
✅ Pesi PointPillars gia presenti in: weights/pointpillar_kitti.pth
📦 Caricamento checkpoint PyTorch da: weights/pointpillar_kitti.pth
✅ Checkpoint PointPillars caricato con successo in memoria!
